In [ ]:
# ========================================
# PLATFORM SETUP — Choose ONE option
# ========================================

# --- OPTION A: KAGGLE (default) ---
import os, sys
KAGGLE_DATASET = '/kaggle/input/brain-tumor-project-files'  # <-- adjust if needed
sys.path.append(KAGGLE_DATASET)
DATA_DIR = os.path.join(KAGGLE_DATASET, 'data', 'classification')
MODEL_DIR = '/kaggle/working/models'
RESULTS_DIR = '/kaggle/working/results'
BASE_DIR = '/kaggle/working'

# --- OPTION B: GOOGLE COLAB (uncomment below, comment out Option A) ---
# from google.colab import drive
# drive.mount('/content/drive')
# import os, sys
# os.chdir('/content/drive/MyDrive/brain-tumor-project/notebooks')
# BASE_DIR = os.path.dirname(os.getcwd())
# sys.path.append(BASE_DIR)
# DATA_DIR = os.path.join(BASE_DIR, 'data', 'classification')
# MODEL_DIR = os.path.join(BASE_DIR, 'models')
# RESULTS_DIR = os.path.join(BASE_DIR, 'results')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Verify data
!ls {DATA_DIR}/train/


# 06 - Brain Tumor Classification with ResNet101V2

**Architecture:** ResNet101V2 pretrained on ImageNet

**Input Size:** 224×224 | **Classes:** 4 | **Dataset:** Merged (~15,000+ images)

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, f1_score
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
import warnings; warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30
MODEL_KEY = 'resnet101'
MODEL_NAME = 'ResNet101V2'

## 1. Data Loading & Augmentation

In [ ]:
from tensorflow.keras.applications.resnet_v2 import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25, width_shift_range=0.15, height_shift_range=0.15,
    shear_range=0.1, zoom_range=0.15, horizontal_flip=True,
    brightness_range=[0.85, 1.15], fill_mode='nearest', validation_split=0.2
)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'train'), target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', subset='training', shuffle=True, seed=42)
val_gen = train_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'train'), target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', subset='validation', shuffle=False, seed=42)
test_gen = test_datagen.flow_from_directory(
    os.path.join(DATA_DIR, 'test'), target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

print(f'Train: {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}')
print(f'Classes: {train_gen.class_indices}')

## 2. Build ResNet101V2 Model (Improved Head + Fine-tuning)

In [ ]:
from tensorflow.keras.applications import ResNet101V2

base_model = ResNet101V2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Unfreeze top 50 layers for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

# Stronger classification head
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(4, activation='softmax')(x)
model = models.Model(inputs, outputs, name='resnet101')

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable = total_params - trainable_params
print(f'Total: {total_params:,} | Trainable: {trainable_params:,} | Non-trainable: {non_trainable:,}')
model.summary()

## 3. Training (with Class Weights)

In [ ]:
class_weights = compute_class_weight('balanced', classes=np.array([0,1,2,3]), y=train_gen.classes)
class_weight_dict = dict(enumerate(class_weights))
print(f'Class weights: {class_weight_dict}')

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'resnet101_best.h5'),
        monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
)

import time; start_time = time.time()
history = model.fit(train_gen, epochs=EPOCHS, validation_data=val_gen,
                    callbacks=callbacks, class_weight=class_weight_dict, verbose=1)
training_time = time.time() - start_time
print(f'\nTraining completed in {training_time/60:.1f} minutes')

## 4. Training History Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], label='Train', color='#FF6B6B', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val', color='#4ECDC4', linewidth=2)
axes[0].set_title('ResNet101V2 - Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=12); axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['loss'], label='Train', color='#FF6B6B', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val', color='#4ECDC4', linewidth=2)
axes[1].set_title('ResNet101V2 - Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=12); axes[1].grid(True, alpha=0.3)
plt.suptitle('ResNet101V2 Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'resnet101_training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Evaluation on Test Set

In [ ]:
best_model_path = os.path.join(MODEL_DIR, 'resnet101_best.h5')
if os.path.exists(best_model_path):
    model = tf.keras.models.load_model(best_model_path)
    print(f'Loaded best model from {best_model_path}')

test_loss, test_acc, test_prec, test_rec, test_auc_score = model.evaluate(test_gen, verbose=1)
print(f'\n{"="*50}')
print('ResNet101V2 - TEST RESULTS')
print(f'{"="*50}')
print(f'Test Accuracy:  {test_acc:.4f}')
print(f'Test Precision: {test_prec:.4f}')
print(f'Test Recall:    {test_rec:.4f}')
print(f'Test AUC:       {test_auc_score:.4f}')
print(f'Test Loss:      {test_loss:.4f}')

## 6. Classification Report & Confusion Matrix

In [ ]:
test_gen.reset()
y_pred_proba = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_gen.classes

print('\nClassification Report:')
print('=' * 60)
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
print(report)
f1 = f1_score(y_true, y_pred, average='weighted')
print(f'Weighted F1-Score: {f1:.4f}')

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('ResNet101V2 - Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'resnet101_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. ROC-AUC Curves

In [ ]:
y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3])
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
for i, (cls, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{cls} (AUC = {roc_auc_val:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ResNet101V2 - ROC Curves', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'resnet101_roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Results

In [ ]:
import json as json_lib
results = {
    'model': MODEL_NAME, 'model_key': MODEL_KEY, 'input_size': IMG_SIZE,
    'total_params': int(total_params), 'trainable_params': int(trainable_params),
    'epochs_trained': len(history.history['loss']),
    'training_time_sec': round(training_time, 1),
    'test_accuracy': round(float(test_acc), 4),
    'test_precision': round(float(test_prec), 4),
    'test_recall': round(float(test_rec), 4),
    'test_auc': round(float(test_auc_score), 4),
    'test_f1': round(float(f1), 4),
    'test_loss': round(float(test_loss), 4),
    'best_val_accuracy': round(max(history.history['val_accuracy']), 4),
}
with open(os.path.join(RESULTS_DIR, 'resnet101_results.json'), 'w') as f:
    json_lib.dump(results, f, indent=2)
print('Results saved to results/resnet101_results.json')
print('Model saved to models/resnet101_best.h5')